In [49]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
import seaborn as sns
plt.style.use('ggplot')
# pd.set_option('max_columns',200)
# pd.set_option('display.max_rows',1000)

In [50]:
df=pd.read_csv("D:/EDA/Cardiovascular_Disease_Dataset.csv")

In [51]:
df.shape

(1000, 14)

In [52]:
df.head()

,patientid,age,gender,chestpain,restingBP,serumcholestrol,fastingbloodsugar,restingrelectro,maxheartrate,exerciseangia,oldpeak,slope,noofmajorvessels,target
0,103368,53,1,2,171,0,0,1,147,0,5.3,3,3,1
1,119250,40,1,0,94,229,0,1,115,0,3.7,1,1,0
2,119372,49,1,2,133,142,0,0,202,1,5.0,1,0,0
3,132514,43,1,0,138,295,1,1,153,0,3.2,2,2,1
4,146211,31,1,1,199,0,0,2,136,0,5.3,3,2,1


In [53]:
df.columns

Index(['patientid', 'age', 'gender', 'chestpain', 'restingBP',
       'serumcholestrol', 'fastingbloodsugar', 'restingrelectro',
       'maxheartrate', 'exerciseangia', 'oldpeak', 'slope', 'noofmajorvessels',
       'target'],
      dtype='object')

In [54]:
df.dtypes

patientid              int64
age                    int64
gender                 int64
chestpain              int64
restingBP              int64
serumcholestrol        int64
fastingbloodsugar      int64
restingrelectro        int64
maxheartrate           int64
exerciseangia          int64
oldpeak              float64
slope                  int64
noofmajorvessels       int64
target                 int64
dtype: object

In [55]:
df=df.rename(columns={
                   'age':'Age',
                   'gender':'Gender',
                   'chestpain':'Chestpain',
                   'restingBP':'Resting_BP',
                   'serumcholestrol':'Serum_cholestrol',
                   'fastingbloodsugar':'Fasting_Blood_Sugar',
                   'restingrelectro': 'Resting_Electrocardiogram',
                   'maxheartrate': 'Max_Heartrate',
                   'exerciseangia': "Exercise_Induced_Angina",
                   'oldpeak':'OldPeak',
                   'slope': 'Slope',
                   'noofmajorvessels': 'Number_of_Major_Vessels',
                   'target':'Target'
                   })

In [56]:
df=df.drop(columns="patientid",axis=1)

In [57]:
df.isna().sum()

Age                          0
Gender                       0
Chestpain                    0
Resting_BP                   0
Serum_cholestrol             0
Fasting_Blood_Sugar          0
Resting_Electrocardiogram    0
Max_Heartrate                0
Exercise_Induced_Angina      0
OldPeak                      0
Slope                        0
Number_of_Major_Vessels      0
Target                       0
dtype: int64

In [58]:
df['Serum_cholestrol']=df['Serum_cholestrol'].replace(0,np.nan)

In [59]:
median_chol=df['Serum_cholestrol'].median()

In [60]:
df['Serum_cholestrol']=df['Serum_cholestrol'].fillna(median_chol)

In [61]:
df.query("Serum_cholestrol == 0")

,Age,Gender,Chestpain,Resting_BP,Serum_cholestrol,Fasting_Blood_Sugar,Resting_Electrocardiogram,Max_Heartrate,Exercise_Induced_Angina,OldPeak,Slope,Number_of_Major_Vessels,Target


In [42]:
df.head()

,Age,Gender,Chestpain,Resting_BP,Serum_cholestrol,Fasting_Blood_Sugar,Resting_Electrocardiogram,Max_Heartrate,Exercise_Induced_Angina,OldPeak,Slope,Number_of_Major_Vessels,Target
0,53,1,2,171,326.0,0,1,147,0,5.3,3,3,1
1,40,1,0,94,229.0,0,1,115,0,3.7,1,1,0
2,49,1,2,133,142.0,0,0,202,1,5.0,1,0,0
3,43,1,0,138,295.0,1,1,153,0,3.2,2,2,1
4,31,1,1,199,326.0,0,2,136,0,5.3,3,2,1


In [62]:
cols_to_check = ['Resting_BP','Serum_cholestrol','Max_Heartrate','OldPeak']
# ============================================================
# Step 1: Detect outliers using IQR
# ============================================================
for col in cols_to_check:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: Q1={Q1}, Q3={Q3}, IQR={IQR}")
    print(f"  Bounds: [{lower:.1f}, {upper:.1f}]")
    print(f"  Outliers found: {len(outliers)}\n")

Resting_BP: Q1=129.0, Q3=181.0, IQR=52.0
  Bounds: [51.0, 259.0]
  Outliers found: 0

Serum_cholestrol: Q1=249.0, Q3=404.25, IQR=155.25
  Bounds: [16.1, 637.1]
  Outliers found: 0

Max_Heartrate: Q1=119.75, Q3=175.0, IQR=55.25
  Bounds: [36.9, 257.9]
  Outliers found: 0

OldPeak: Q1=1.3, Q3=4.1, IQR=2.8
  Bounds: [-2.9, 8.3]
  Outliers found: 0



In [63]:
# Cap/Clip outliers
for col in cols_to_check:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = len(df[(df[col] < lower) | (df[col] > upper)])
    df[col] = df[col].clip(lower=lower, upper=upper)
    print(f"{col}: {before} outliers capped to [{lower:.1f}, {upper:.1f}]")


Resting_BP: 0 outliers capped to [51.0, 259.0]
Serum_cholestrol: 0 outliers capped to [16.1, 637.1]
Max_Heartrate: 0 outliers capped to [36.9, 257.9]
OldPeak: 0 outliers capped to [-2.9, 8.3]


In [ ]:
# from sklearn.preprocessing import StandardScaler

# cols_to_scale=['Age','Resting_BP','Serum_cholestrol','Max_Heartrate','OldPeak']

# scaler=StandardScaler()

# df[cols_to_scale]=scaler.fit_transform(df[cols_to_scale])

In [64]:
df.head()

,Age,Gender,Chestpain,Resting_BP,Serum_cholestrol,Fasting_Blood_Sugar,Resting_Electrocardiogram,Max_Heartrate,Exercise_Induced_Angina,OldPeak,Slope,Number_of_Major_Vessels,Target
0,53,1,2,171,326.0,0,1,147,0,5.3,3,3,1
1,40,1,0,94,229.0,0,1,115,0,3.7,1,1,0
2,49,1,2,133,142.0,0,0,202,1,5.0,1,0,0
3,43,1,0,138,295.0,1,1,153,0,3.2,2,2,1
4,31,1,1,199,326.0,0,2,136,0,5.3,3,2,1


In [65]:
df.to_csv('preprocessed_data.csv',index=False)